In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

print("All libraries imported successfully!")

All libraries imported successfully!


In [2]:
path = "D:/e-commerce-analytics-project/data/raw/archive8/"

orders = pd.read_csv(path + "olist_orders_dataset.csv")
customers = pd.read_csv(path + "olist_customers_dataset.csv")
order_items = pd.read_csv(path + "olist_order_items_dataset.csv")
payments = pd.read_csv(path + "olist_order_payments_dataset.csv")
reviews = pd.read_csv(path + "olist_order_reviews_dataset.csv")
products = pd.read_csv(path + "olist_products_dataset.csv")
sellers = pd.read_csv(path + "olist_sellers_dataset.csv")
geolocation = pd.read_csv(path + "olist_geolocation_dataset.csv")
category = pd.read_csv(path + "product_category_name_translation.csv")

print("All 9 files loaded successfully!")

All 9 files loaded successfully!


In [3]:
print("orders:", orders.shape)
print("customers:", customers.shape)
print("order_items:", order_items.shape)
print("payments:", payments.shape)
print("reviews:", reviews.shape)
print("products:", products.shape)
print("sellers:", sellers.shape)
print("geolocation:", geolocation.shape)
print("category:", category.shape)

orders: (99441, 8)
customers: (99441, 5)
order_items: (112650, 7)
payments: (103886, 5)
reviews: (99224, 7)
products: (32951, 9)
sellers: (3095, 4)
geolocation: (1000163, 5)
category: (71, 2)


In [4]:
print("=== MISSING VALUES ===\n")
print("Orders:\n", orders.isnull().sum())
print("\nCustomers:\n", customers.isnull().sum())
print("\nProducts:\n", products.isnull().sum())
print("\nReviews:\n", reviews.isnull().sum())

=== MISSING VALUES ===

Orders:
 order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

Customers:
 customer_id                 0
customer_unique_id          0
customer_zip_code_prefix    0
customer_city               0
customer_state              0
dtype: int64

Products:
 product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

Reviews:
 review_id                      0
order_id                       0
review_score                   0
review_comment_title       87656


In [5]:
print("=== data type ===")
print(orders.dtypes)

=== data type ===
order_id                         object
customer_id                      object
order_status                     object
order_purchase_timestamp         object
order_approved_at                object
order_delivered_carrier_date     object
order_delivered_customer_date    object
order_estimated_delivery_date    object
dtype: object


In [6]:
date_columns = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col])

print("Dates fixed!")
print(orders.dtypes)

Dates fixed!
order_id                                 object
customer_id                              object
order_status                             object
order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object


In [7]:
# Calculate delivery delay in days
orders['delivery_delay_days'] = (
    orders['order_delivered_customer_date'] - 
    orders['order_estimated_delivery_date']
).dt.days

# Create order month column
orders['order_month'] = (
    orders['order_purchase_timestamp'].dt.to_period('M')
)

# Create late delivery flag
orders['is_late'] = orders['delivery_delay_days'] > 0

print("New columns created!")
print(orders[['delivery_delay_days', 
              'order_month', 
              'is_late']].head())

New columns created!
   delivery_delay_days order_month  is_late
0                 -8.0     2017-10    False
1                 -6.0     2018-07    False
2                -18.0     2018-08    False
3                -13.0     2017-11    False
4                -10.0     2018-02    False


In [8]:
# Fix orders missing values
# Keep blank dates as is (they are canceled orders)
# Just verify
print("orders missing values")
print(orders.isnull().sum())

# Fix products missing values
products_clean=products.dropna(
    subset=['product_category_name']
)

#fix reviews missing value

reviews['review_comment_title'] = (
    reviews['review_comment_title'].fillna('No comment')
)

reviews['review_comment_message'] = (
    reviews['review_comment_message'].fillna('No comment')
)

print("\nProducts before cleaning:", products.shape)
print("Products after cleaning:", products_clean.shape)
print("\nReviews missing after fix:")
print(reviews[['review_comment_title',
               'review_comment_message']].isnull().sum())
print("\nMissing values handled!")

orders missing values
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
delivery_delay_days              2965
order_month                         0
is_late                             0
dtype: int64

Products before cleaning: (32951, 9)
Products after cleaning: (32341, 9)

Reviews missing after fix:
review_comment_title      0
review_comment_message    0
dtype: int64

Missing values handled!


In [9]:
# Save cleaned files to data/cleaned/ folder
cleaned_path = "D:/e-commerce-analytics-project/data/cleaned/"
orders.to_csv(cleaned_path + "orders_cleaned.csv",index=False)
customers.to_csv(cleaned_path + "customers_cleaned.csv",index=False)
products_clean.to_csv(cleaned_path + "products_cleaned.csv",index=False)
reviews.to_csv(cleaned_path + "reviews_cleaned.csv",index=False)
order_items.to_csv(cleaned_path + "order_items_cleaned.csv",index=False)
payments.to_csv(cleaned_path + "payments_cleaned.csv",index=False)
sellers.to_csv(cleaned_path + "sellers_cleaned.csv",index=False)

print("All cleaned files exported successfully!")


All cleaned files exported successfully!


In [10]:
from sqlalchemy import create_engine

engine = create_engine(
    "mssql+pyodbc://MKS/EcommerceAnalytics"
    "?driver=ODBC+Driver+17+for+SQL+Server"
    "&trusted_connection=yes"
    "&TrustServerCertificate=yes"
)

try:
    with engine.connect() as conn:
        print("Connected successfully!")
except Exception as e:
    print("Error:", e)


C:\Users\megha\AppData\Local\Temp\ipykernel_27800\2970177281.py:11: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  with engine.connect() as conn:


Connected successfully!


In [11]:
import pyodbc

conn_str = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=MKS;"
    "DATABASE=EcommerceAnalytics;"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

try:
    conn = pyodbc.connect(conn_str)
    print("Connected successfully!")
except Exception as e:
    print("Error:", e)

Connected successfully!


In [12]:
# Fix columns before SQL import
orders['order_month'] = orders['order_month'].astype(str)
orders['is_late'] = orders['is_late'].astype(str)

print("Columns fixed! Ready to import!")

Columns fixed! Ready to import!


In [13]:
# Import all 7 tables to SQL Server
from sqlalchemy import create_engine

engine = create_engine(
    "mssql+pyodbc://MKS/EcommerceAnalytics"
    "?driver=ODBC+Driver+17+for+SQL+Server"
    "&trusted_connection=yes"
    "&TrustServerCertificate=yes"
)

orders.to_sql('orders', engine, 
              if_exists='replace', index=False)
print("1/7 orders imported!")

customers.to_sql('customers', engine, 
                 if_exists='replace', index=False)
print("2/7 customers imported!")

order_items.to_sql('order_items', engine, 
                   if_exists='replace', index=False)
print("3/7 order_items imported!")

payments.to_sql('payments', engine, 
                if_exists='replace', index=False)
print("4/7 payments imported!")

sellers.to_sql('sellers', engine, 
               if_exists='replace', index=False)
print("5/7 sellers imported!")

products_clean.to_sql('products', engine, 
                      if_exists='replace', index=False)
print("6/7 products imported!")

reviews.to_sql('reviews', engine, 
               if_exists='replace', index=False)
print("7/7 reviews imported!")
print("\nAll tables imported to SQL Server! 🎉")

C:\Users\megha\anaconda3\Lib\site-packages\pandas\io\sql.py:1648: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())


1/7 orders imported!
2/7 customers imported!
3/7 order_items imported!
4/7 payments imported!
5/7 sellers imported!
6/7 products imported!
7/7 reviews imported!

All tables imported to SQL Server! 🎉
